````markdown
# Dataset Details

## Dataset

- **Base Dataset:** Google Speech Commands v0.02
- **Task:** Keyword Spotting / Wake-Word Detection
- **Target Wake Word:** `marvin`
- **Audio Sample Rate:** 16 kHz
- **Audio Duration:** 1 second
- **Audio Channels:** Mono
- **Total Processed Samples:** 4,000

## Dataset Classes

| Class ID | Class | Number of Samples |
|---:|---|---:|
| 0 | Marvin | 1,000 |
| 1 | Unknown Speech | 2,000 |
| 2 | Background | 1,000 |
| **Total** | | **4,000** |

### Class 0 — Marvin

- Target keyword class.
- 1,000 samples selected from the `marvin` category.

### Class 1 — Unknown Speech

The following words were used as non-target speech:

- yes
- no
- up
- down
- left
- right
- on
- off
- stop
- go
- zero
- one
- two
- three
- four
- five
- six
- seven
- eight
- nine
- bed
- bird
- cat
- dog
- happy
- house
- sheila
- tree
- wow
- backward
- forward
- follow
- learn
- visual

- **Total samples:** 2,000

### Class 2 — Background

- Background-noise recordings from `_background_noise_`.
- Random 1-second segments were extracted from the long background recordings.
- **Total samples:** 1,000

## Audio Preprocessing

Each audio sample is converted to:

- **Sample Rate:** 16,000 Hz
- **Duration:** 1 second
- **Number of Samples:** 16,000
- **Channels:** Mono

Audio shorter than 1 second is zero-padded, while audio longer than 1 second is truncated.

## MFCC Feature Extraction

MFCC features are extracted using:

| Parameter | Value |
|---|---:|
| Sample Rate | 16,000 Hz |
| Number of MFCCs | 13 |
| FFT Size (`N_FFT`) | 512 |
| Hop Length | 320 |
| Time Frames | 49 |

The MFCC representation is converted from:

```text
(13, 49)
````

to:

```text
(49, 13, 1)
```

Therefore, the neural-network input shape is:

```text
(49, 13, 1)
```

## Dataset Split

A stratified **80/10/10** split was used:

| Dataset    |   Samples |
| ---------- | --------: |
| Training   |     3,200 |
| Validation |       400 |
| Testing    |       400 |
| **Total**  | **4,000** |

### Training Set

```text
Marvin       : 800
Unknown      : 1600
Background   : 800
Total        : 3200
```

### Validation Set

```text
Marvin       : 100
Unknown      : 200
Background   : 100
Total        : 400
```

### Test Set

```text
Marvin       : 100
Unknown      : 200
Background   : 100
Total        : 400
```

## Processed Dataset Files

```text
C:\hello_wakey\processed\
│
├── X_train.npy
├── y_train.npy
├── X_val.npy
├── y_val.npy
├── X_test.npy
└── y_test.npy
```

### Feature Shapes

```text
X_train → (3200, 49, 13, 1)
X_val   → (400, 49, 13, 1)
X_test  → (400, 49, 13, 1)
```

### Label Shapes

```text
y_train → (3200,)
y_val   → (400,)
y_test  → (400,)
```

### Label Mapping

```text
0 → Marvin
1 → Unknown Speech
2 → Background
```

## Dataset Purpose

The dataset is designed to train a lightweight wake-word detector that learns to distinguish:

```text
Marvin
   vs
Other Speech
   vs
Background Noise
```

The current `marvin` keyword is being used as a temporary target to validate the complete wake-word detection pipeline before replacing it with the final custom wake word, `Hello Wakey`.

## Dataset Selection Objective

The processed dataset is used to evaluate multiple DS-CNN architectures with different parameter counts.

The objective is to determine the **smallest model that provides stable wake-word detection performance** while maintaining the required embedded-system constraints.

## Current Experimental Limitation

The current 80/10/10 split is a **sample-level random stratified split**.

For the final research evaluation, a **speaker-independent split** should be used so that speakers in the test set are not also present in the training set.

```
```


In [ ]:
import os
import json
import numpy as np
import tensorflow as tf

from pathlib import Path
from sklearn.metrics import (
    confusion_matrix,
    classification_report,
    accuracy_score,
    precision_recall_fscore_support
)


# ============================================================
# PROJECT DIRECTORY
# ============================================================

# Automatically find the folder containing this Python script
PROJECT_DIR = Path(__file__).resolve().parent


# ============================================================
# CONFIGURATION
# ============================================================

DATA_DIR = PROJECT_DIR / "processed"

OUTPUT_DIR = PROJECT_DIR / "models"

RANDOM_SEED = 42

np.random.seed(RANDOM_SEED)
tf.random.set_seed(RANDOM_SEED)


# ============================================================
# TRAINING CONFIGURATION
# ============================================================

EPOCHS = 40
BATCH_SIZE = 32


# ============================================================
# MODEL CONFIGURATIONS
# ============================================================

MODEL_CONFIGS = {

    # --------------------------------------------------------
    # Very small
    # --------------------------------------------------------

    "tiny": {
        "channels": [8, 12, 16],
        "dense": 16
    },

    # --------------------------------------------------------
    # Small
    # --------------------------------------------------------

    "small": {
        "channels": [12, 16, 24],
        "dense": 24
    },

    # --------------------------------------------------------
    # Medium
    # --------------------------------------------------------

    "medium": {
        "channels": [16, 24, 32],
        "dense": 32
    },

    # --------------------------------------------------------
    # Large
    # --------------------------------------------------------

    "large": {
        "channels": [24, 32, 48],
        "dense": 48
    },

    # --------------------------------------------------------
    # Very large
    # --------------------------------------------------------

    "xlarge": {
        "channels": [32, 48, 64],
        "dense": 64
    }
}


# ============================================================
# OUTPUT DIRECTORY
# ============================================================

OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True
)


# ============================================================
# LOAD DATASET
# ============================================================

print("=" * 60)
print("LOADING DATASET")
print("=" * 60)


X_train = np.load(
    DATA_DIR / "X_train.npy"
)

y_train = np.load(
    DATA_DIR / "y_train.npy"
)

X_val = np.load(
    DATA_DIR / "X_val.npy"
)

y_val = np.load(
    DATA_DIR / "y_val.npy"
)

X_test = np.load(
    DATA_DIR / "X_test.npy"
)

y_test = np.load(
    DATA_DIR / "y_test.npy"
)


print("\nDataset shapes:")

print(
    "X_train:",
    X_train.shape
)

print(
    "y_train:",
    y_train.shape
)

print(
    "X_val:",
    X_val.shape
)

print(
    "y_val:",
    y_val.shape
)

print(
    "X_test:",
    X_test.shape
)

print(
    "y_test:",
    y_test.shape
)


# ============================================================
# NORMALIZATION
# ============================================================

# MFCC values are typically negative/positive.
#
# We calculate normalization statistics ONLY from
# the training set.
#
# Validation and test use the same statistics.
# ============================================================

mean = np.mean(X_train)

std = np.std(X_train)


print("\nMFCC normalization:")
print("Mean:", mean)
print("Std :", std)


# ============================================================
# SAVE NORMALIZATION PARAMETERS
# ============================================================

normalization_path = OUTPUT_DIR / "normalization.json"

with open(
    normalization_path,
    "w"
) as f:

    json.dump(
        {
            "mean": float(mean),
            "std": float(std)
        },
        f,
        indent=4
    )


print("\nNormalization parameters saved to:")
print(normalization_path)


# ============================================================
# APPLY NORMALIZATION
# ============================================================

X_train = (
    X_train - mean
) / (std + 1e-8)

X_val = (
    X_val - mean
) / (std + 1e-8)

X_test = (
    X_test - mean
) / (std + 1e-8)


# ============================================================
# DS-CNN MODEL
# ============================================================

def depthwise_separable_block(
    x,
    channels,
    name
):

    # --------------------------------------------------------
    # Depthwise convolution
    # --------------------------------------------------------

    x = tf.keras.layers.DepthwiseConv2D(
        kernel_size=(3, 3),
        padding="same",
        use_bias=True,
        name=f"{name}_dw"
    )(x)

    x = tf.keras.layers.BatchNormalization(
        name=f"{name}_dw_bn"
    )(x)

    x = tf.keras.layers.ReLU(
        name=f"{name}_dw_relu"
    )(x)


    # --------------------------------------------------------
    # Pointwise convolution
    # --------------------------------------------------------

    x = tf.keras.layers.Conv2D(
        filters=channels,
        kernel_size=(1, 1),
        padding="same",
        use_bias=True,
        name=f"{name}_pw"
    )(x)

    x = tf.keras.layers.BatchNormalization(
        name=f"{name}_pw_bn"
    )(x)

    x = tf.keras.layers.ReLU(
        name=f"{name}_pw_relu"
    )(x)

    return x


# ============================================================
# BUILD MODEL
# ============================================================

def build_model(
    channels,
    dense_units
):

    c1, c2, c3 = channels

    inputs = tf.keras.Input(
        shape=(49, 13, 1),
        name="mfcc_input"
    )


    # --------------------------------------------------------
    # Initial convolution
    # --------------------------------------------------------

    x = tf.keras.layers.Conv2D(
        filters=c1,
        kernel_size=(3, 3),
        padding="same",
        use_bias=True,
        name="initial_conv"
    )(inputs)

    x = tf.keras.layers.BatchNormalization(
        name="initial_bn"
    )(x)

    x = tf.keras.layers.ReLU(
        name="initial_relu"
    )(x)


    # --------------------------------------------------------
    # DS Block 1
    # --------------------------------------------------------

    x = depthwise_separable_block(
        x,
        c1,
        "block1"
    )


    # --------------------------------------------------------
    # DS Block 2
    # --------------------------------------------------------

    x = depthwise_separable_block(
        x,
        c2,
        "block2"
    )


    # --------------------------------------------------------
    # DS Block 3
    # --------------------------------------------------------

    x = depthwise_separable_block(
        x,
        c3,
        "block3"
    )


    # --------------------------------------------------------
    # Global Average Pooling
    # --------------------------------------------------------

    x = tf.keras.layers.GlobalAveragePooling2D(
        name="global_average_pooling"
    )(x)


    # --------------------------------------------------------
    # Dense layer
    # --------------------------------------------------------

    x = tf.keras.layers.Dense(
        dense_units,
        activation="relu",
        name="dense"
    )(x)


    # --------------------------------------------------------
    # Output
    # --------------------------------------------------------

    outputs = tf.keras.layers.Dense(
        3,
        activation="softmax",
        name="output"
    )(x)


    model = tf.keras.Model(
        inputs=inputs,
        outputs=outputs
    )

    return model


# ============================================================
# TRAIN ONE MODEL
# ============================================================

def train_model(
    model_name,
    config
):

    print("\n")
    print("=" * 60)
    print(
        f"TRAINING MODEL: {model_name.upper()}"
    )
    print("=" * 60)

    channels = config["channels"]
    dense_units = config["dense"]


    # --------------------------------------------------------
    # Build
    # --------------------------------------------------------

    model = build_model(
        channels=channels,
        dense_units=dense_units
    )


    # --------------------------------------------------------
    # Parameter count
    # --------------------------------------------------------

    total_params = model.count_params()

    trainable_params = np.sum([
        np.prod(v.shape)
        for v in model.trainable_variables
    ])

    non_trainable_params = (
        total_params -
        trainable_params
    )


    print("\nArchitecture:")

    print(
        "Channels:",
        channels
    )

    print(
        "Dense:",
        dense_units
    )

    print(
        "Total parameters:",
        total_params
    )

    print(
        "Trainable parameters:",
        trainable_params
    )

    print(
        "Non-trainable parameters:",
        non_trainable_params
    )


    # --------------------------------------------------------
    # Compile
    # --------------------------------------------------------

    model.compile(

        optimizer=tf.keras.optimizers.Adam(
            learning_rate=0.001
        ),

        loss=tf.keras.losses.SparseCategoricalCrossentropy(),

        metrics=[
            "accuracy"
        ]
    )


    # --------------------------------------------------------
    # Model save path
    # --------------------------------------------------------

    model_path = (
        OUTPUT_DIR /
        f"{model_name}.keras"
    )


    # --------------------------------------------------------
    # Callbacks
    # --------------------------------------------------------

    callbacks = [

        tf.keras.callbacks.EarlyStopping(

            monitor="val_loss",

            patience=7,

            restore_best_weights=True
        ),

        tf.keras.callbacks.ReduceLROnPlateau(

            monitor="val_loss",

            factor=0.5,

            patience=3,

            min_lr=1e-6
        ),

        tf.keras.callbacks.ModelCheckpoint(

            filepath=model_path,

            monitor="val_accuracy",

            save_best_only=True,

            mode="max"
        )
    ]


    # --------------------------------------------------------
    # Training
    # --------------------------------------------------------

    history = model.fit(

        X_train,
        y_train,

        validation_data=(
            X_val,
            y_val
        ),

        epochs=EPOCHS,

        batch_size=BATCH_SIZE,

        shuffle=True,

        callbacks=callbacks,

        verbose=1
    )


    # --------------------------------------------------------
    # Reload best model
    # --------------------------------------------------------

    model = tf.keras.models.load_model(
        model_path
    )


    # ========================================================
    # TEST
    # ========================================================

    print("\n")
    print("-" * 60)
    print(
        f"TESTING MODEL: {model_name}"
    )
    print("-" * 60)


    test_loss, test_accuracy = model.evaluate(

        X_test,
        y_test,

        batch_size=BATCH_SIZE,

        verbose=0
    )


    # --------------------------------------------------------
    # Predictions
    # --------------------------------------------------------

    probabilities = model.predict(

        X_test,

        batch_size=BATCH_SIZE,

        verbose=0
    )

    y_pred = np.argmax(
        probabilities,
        axis=1
    )


    # ========================================================
    # METRICS
    # ========================================================

    accuracy = accuracy_score(
        y_test,
        y_pred
    )


    precision, recall, f1, support = (
        precision_recall_fscore_support(

            y_test,
            y_pred,

            labels=[0, 1, 2],

            zero_division=0
        )
    )


    # --------------------------------------------------------
    # Confusion matrix
    # --------------------------------------------------------

    cm = confusion_matrix(
        y_test,
        y_pred,

        labels=[0, 1, 2]
    )


    print("\nConfusion Matrix:")

    print(
        "             Predicted"
    )

    print(
        "             Marvin Unknown Background"
    )

    print(
        f"Marvin      {cm[0,0]:8d} {cm[0,1]:7d} {cm[0,2]:10d}"
    )

    print(
        f"Unknown     {cm[1,0]:8d} {cm[1,1]:7d} {cm[1,2]:10d}"
    )

    print(
        f"Background  {cm[2,0]:8d} {cm[2,1]:7d} {cm[2,2]:10d}"
    )


    # ========================================================
    # PER CLASS
    # ========================================================

    print("\nPer-class metrics:")

    class_names = [
        "marvin",
        "unknown",
        "background"
    ]


    for i, name in enumerate(
        class_names
    ):

        print(
            f"{name:<12} "
            f"Precision={precision[i]:.4f} "
            f"Recall={recall[i]:.4f} "
            f"F1={f1[i]:.4f}"
        )


    # ========================================================
    # SAVE CLASSIFICATION REPORT
    # ========================================================

    report = classification_report(

        y_test,
        y_pred,

        labels=[0, 1, 2],

        target_names=class_names,

        zero_division=0
    )


    print("\nClassification report:")
    print(report)


    # ========================================================
    # MARVIN-SPECIFIC METRICS
    # ========================================================

    marvin_recall = recall[0]

    marvin_precision = precision[0]

    marvin_f1 = f1[0]


    # --------------------------------------------------------
    # False positive rate for Marvin
    #
    # Non-Marvin samples incorrectly classified as Marvin
    # --------------------------------------------------------

    non_marvin = (
        (y_test == 1) |
        (y_test == 2)
    )

    false_marvin = (
        (y_pred == 0) &
        non_marvin
    )

    true_non_marvin = (
        (y_pred != 0) &
        non_marvin
    )

    false_positive_rate = (

        np.sum(false_marvin) /

        (
            np.sum(false_marvin) +
            np.sum(true_non_marvin)
        )
    )


    print("\nMarvin-specific metrics:")

    print(
        f"Marvin recall : {marvin_recall:.4f}"
    )

    print(
        f"Marvin precision: {marvin_precision:.4f}"
    )

    print(
        f"Marvin F1     : {marvin_f1:.4f}"
    )

    print(
        f"Marvin FPR    : {false_positive_rate:.6f}"
    )


    # ========================================================
    # BEST VALIDATION ACCURACY
    # ========================================================

    best_val_accuracy = max(
        history.history["val_accuracy"]
    )

    best_train_accuracy = max(
        history.history["accuracy"]
    )


    # ========================================================
    # SAVE RESULTS
    # ========================================================

    result = {

        "model": model_name,

        "channels": channels,

        "dense_units": dense_units,

        "total_parameters": int(
            total_params
        ),

        "trainable_parameters": int(
            trainable_params
        ),

        "non_trainable_parameters": int(
            non_trainable_params
        ),

        "best_train_accuracy": float(
            best_train_accuracy
        ),

        "best_validation_accuracy": float(
            best_val_accuracy
        ),

        "test_accuracy": float(
            accuracy
        ),

        "marvin_precision": float(
            marvin_precision
        ),

        "marvin_recall": float(
            marvin_recall
        ),

        "marvin_f1": float(
            marvin_f1
        ),

        "marvin_false_positive_rate": float(
            false_positive_rate
        ),

        "confusion_matrix": cm.tolist(),

        "epochs_trained": len(
            history.history["loss"]
        )
    }


    result_path = (
        OUTPUT_DIR /
        f"{model_name}_results.json"
    )


    with open(
        result_path,
        "w"
    ) as f:

        json.dump(
            result,
            f,
            indent=4
        )


    # ========================================================
    # PRINT SUMMARY
    # ========================================================

    print("\n")
    print("=" * 60)
    print(
        f"{model_name.upper()} COMPLETE"
    )
    print("=" * 60)

    print(
        "Parameters:",
        total_params
    )

    print(
        "Best validation accuracy:",
        f"{best_val_accuracy:.4f}"
    )

    print(
        "Test accuracy:",
        f"{accuracy:.4f}"
    )

    print(
        "Marvin precision:",
        f"{marvin_precision:.4f}"
    )

    print(
        "Marvin recall:",
        f"{marvin_recall:.4f}"
    )

    print(
        "Marvin F1:",
        f"{marvin_f1:.4f}"
    )

    print(
        "Marvin FPR:",
        f"{false_positive_rate:.6f}"
    )

    print(
        "Saved model:",
        model_path
    )


    return result


# ============================================================
# TRAIN ALL MODELS
# ============================================================

all_results = []


for model_name, config in MODEL_CONFIGS.items():

    result = train_model(
        model_name,
        config
    )

    all_results.append(
        result
    )


# ============================================================
# FINAL COMPARISON
# ============================================================

print("\n\n")
print("=" * 90)
print("FINAL MODEL COMPARISON")
print("=" * 90)


print(
    f"{'Model':<10}"
    f"{'Params':>10}"
    f"{'Val Acc':>12}"
    f"{'Test Acc':>12}"
    f"{'Marvin P':>12}"
    f"{'Marvin R':>12}"
    f"{'Marvin F1':>12}"
    f"{'Marvin FPR':>14}"
)


print("-" * 90)


for r in all_results:

    print(
        f"{r['model']:<10}"
        f"{r['total_parameters']:>10}"
        f"{r['best_validation_accuracy']:>12.4f}"
        f"{r['test_accuracy']:>12.4f}"
        f"{r['marvin_precision']:>12.4f}"
        f"{r['marvin_recall']:>12.4f}"
        f"{r['marvin_f1']:>12.4f}"
        f"{r['marvin_false_positive_rate']:>14.6f}"
    )


# ============================================================
# SAVE ALL RESULTS
# ============================================================

summary_path = (
    OUTPUT_DIR /
    "all_results.json"
)


with open(
    summary_path,
    "w"
) as f:

    json.dump(
        all_results,
        f,
        indent=4
    )


print("\n")
print(
    "All results saved to:"
)

print(
    summary_path
)

print("\n")
print("TRAINING COMPLETE!")

```markdown
# Model Parameters

MODEL_CONFIGS = {
    "tiny":   {"channels": [8, 12, 16], "dense": 16},
    "small":  {"channels": [12, 16, 24], "dense": 24},
    "medium": {"channels": [16, 24, 32], "dense": 32},
    "medium2":{"channels": [16, 24, 40], "dense": 32},
    "large":  {"channels": [24, 32, 48], "dense": 48},
    "xlarge": {"channels": [32, 48, 64], "dense": 64},
}

# Training Parameters

LEARNING_RATES = [
    0.001,
    0.0005,
    0.0003,
    0.0001
]

BATCH_SIZES = [
    16,
    32,
    64
]

EPOCHS = 40

# MFCC Parameters

SAMPLE_RATE = 16000

N_MFCC = 13

N_FFT = 512

HOP_LENGTH = 320

# Wake-word Threshold

THRESHOLDS = [
    0.50,
    0.60,
    0.70,
    0.80,
    0.90,
    0.95
]

# Temporal Smoothing

CONSECUTIVE_DETECTIONS = [
    1,
    2,
    3
]
```
